# 00. TS3003c.txt 기준 주제 경계 후보 탐지 결과 확인
### * 이 부분은 실험 전 테스트한 결과로, 아래 "01. 20개 회의록 기준 경계 후보 탐지 - 최적 조합 테스트" 부분 이후로 확인하면 됨

## 00-01. 주제 경계 후보 탐지를 위한 청킹

#### 활용 데이터 : TS3003c.txt

In [ ]:
# --------------------------------------------------------------------------------------------------
# 1-0. 설정 
# 경로/파라미터는 해당 셀에서만 수정
# sentence-transformers 신규 설치
# 길이는 모두 글자 수(len, 공백·줄바꿈 포함) 기준
# --------------------------------------------------------------------------------------------------
from pathlib import Path

# ── 경로 ──────────────────────────────────────────────
BASE_DIR = Path.cwd().parent                       # playground/chaewon
INPUT_PATH = BASE_DIR / "dataset" / "TS3003c.txt"  # 청킹할 전사문
OUTPUT_DIR = BASE_DIR / "outputs" / "chunks"       # 결과 저장 폴더 (outputs/는 .gitignore 대상)

# ── 전사문 형식 ───────────────────────────────────────
HEADER_LINES = 3            # 상단 메타데이터 줄 수 (회의 대주제 / 참여자 / 일시) → 청킹 제외
UTTERANCE_SEPARATOR = "\n"  # 청크 안에서 발화를 이어 붙일 구분자

# ── 청킹 파라미터 (글자 수 기준) ─────────────────────────
CHUNK_SIZE = 200                # 청크당 최대 글자 수
CHUNK_OVERLAP = 100             # 인접 청크가 공유할 최대 글자 수 (발화 단위로 맞춤)

# 임베딩 모델
EMBEDDING_MODEL = "alphabrothers/alpha-sts-v0"

# 파라미터 비교용 후보 (마지막 셀, 글자 수 기준)
COMPARE_CONFIGS = [(200, 50), (200, 100), (300, 50), (300, 100), (400, 50), (400, 100), (500, 50), (500, 100)]

In [ ]:
# --------------------------------------------------------------------------------------------------
# 1-1. 전사문 읽기 - 머리말, 발화 분리
# --------------------------------------------------------------------------------------------------
import json

assert 0 <= CHUNK_OVERLAP < CHUNK_SIZE, "CHUNK_OVERLAP은 0 이상, CHUNK_SIZE 미만이어야 합니다."


def load_transcript(path, header_lines):
    lines = path.read_text(encoding="utf-8-sig").splitlines()
    header = {}
    for line in lines[:header_lines]:
        key, _, value = line.partition(":")
        header[key.strip()] = value.strip()
    utterances = [line.strip() for line in lines[header_lines:] if line.strip()]
    return header, utterances


header, utterances = load_transcript(INPUT_PATH, HEADER_LINES)
meeting_id = INPUT_PATH.stem

print("머리말 (청킹 제외):", json.dumps(header, ensure_ascii=False, indent=2))
print(f"발화 수: {len(utterances)}")
print("첫 발화 5개:", utterances[:5])

In [ ]:
# --------------------------------------------------------------------------------------------------
# 1-2. 글자 수 기준 발화 단위 청킹
# --------------------------------------------------------------------------------------------------
def count_chars(text):
    # 글자 수 = len(text) → 공백과 발화 사이 줄바꿈(UTTERANCE_SEPARATOR)도 1자로 셈
    return len(text)


def chunk_utterances(utterances, chunk_size, chunk_overlap, sep=UTTERANCE_SEPARATOR):
    chunks = []
    start = 0
    while start < len(utterances):
        # 1) chunk_size를 넘기 직전까지 발화를 쌓는다 (발화 1개가 chunk_size보다 길면 그 발화 하나로 청크 구성)
        end = start + 1
        while end < len(utterances) and count_chars(sep.join(utterances[start:end + 1])) <= chunk_size:
            end += 1
        text = sep.join(utterances[start:end])
        chunks.append({
            "text": text,
            "metadata": {
                "chunk_index": len(chunks),
                "char_count": count_chars(text),
                "utterance_start": start,    # 포함
                "utterance_end": end - 1,    # 포함
            },
        })
        if end == len(utterances):
            break
        # 2) 청크 끝에서 chunk_overlap 이내의 발화를 다음 청크 시작으로 (항상 앞으로 진행)
        next_start = end
        while next_start - 1 > start and count_chars(sep.join(utterances[next_start - 1:end])) <= chunk_overlap:
            next_start -= 1
        start = next_start
    return chunks


chunks = chunk_utterances(utterances, CHUNK_SIZE, CHUNK_OVERLAP)
print(f"청크 수: {len(chunks)}  (chunk_size={CHUNK_SIZE}, chunk_overlap={CHUNK_OVERLAP})")

In [ ]:
# --------------------------------------------------------------------------------------------------
# 1-3. 결과 확인
# --------------------------------------------------------------------------------------------------
import pandas as pd
from IPython.display import display

chunks_df = pd.DataFrame([{**c["metadata"], "preview": c["text"][:60].replace("\n", " / ")} for c in chunks])
display(chunks_df)
display(chunks_df["char_count"].describe())

# 정합성 검사: 순서, 글자 수 한도(긴 단일 발화 예외), 모든 발화 포함 여부
assert chunks_df["chunk_index"].tolist() == list(range(len(chunks)))
single = chunks_df["utterance_start"] == chunks_df["utterance_end"]
assert ((chunks_df["char_count"] <= CHUNK_SIZE) | single).all()
covered = set()
for c in chunks:
    covered.update(range(c["metadata"]["utterance_start"], c["metadata"]["utterance_end"] + 1))
assert covered == set(range(len(utterances))), "누락된 발화가 있습니다."

print("\n[chunk 0]\n" + chunks[0]["text"])
print("\n[chunk 1] (앞부분이 chunk 0 끝과 겹침)\n" + chunks[1]["text"])

In [ ]:
# --------------------------------------------------------------------------------------------------
# 1-4. 저장
# 현재는 각 단계별로 저장하지 않도록 함
# 만약, 단계별로 저장하고 싶다면 주석 풀고 실행하면 "outputs/chunks" 디렉토리 내 <회의ID>_chunks.json 파일로 저장됨
# utterances도 함께 저장하여 utteraces_start~utterance_end로 [발화번호] 텍스트 형태의 경계 앞뒤 문맥을 다시 만들 수 있음
# output에는 크게 아래와 같은 정보를 포함함
# > 회의ID / metadata(회의주제, 참여자, 일시) / chunking 정보 / utterances 정보(발화별 순서 인덱스) / chunks(각 청킹값)

# chunks 예) CHUNK_SIZE=200일 때
# {'text': '좋아요. 어\n안녕하세요.\n안녕하세요.\n벌써 세 번째 회의네요.\n점심 맛있게 드셨길 바라요.\n저는 아무튼 맛있게 먹었어요.\n음, 어디 보자.\n발표 3.\n좋아요\n이건 음, 오늘 논의할 어, 두 번째 단계예요.\n콘셉트 설계 회의예요.\n그리고 이번 회의의 관심사 몇 가지는 음, 부품의 콘셉트 명세예요. 어, 디자인의 콘셉트 명세. 그리고 트렌드 관찰이요.',
#    'metadata': {'chunk_index': 0,
#     'char_count': 195,
#     'utterance_start': 0,
#     'utterance_end': 11}}
# --------------------------------------------------------------------------------------------------
chunk_result = {
    "meeting_id": meeting_id,
    "metadata": header,
    "chunking": {
        "unit": "character (len, spaces/newlines included)",
        "strategy": "utterance packing; never split inside an utterance",
        "chunk_size": CHUNK_SIZE,
        "chunk_overlap": CHUNK_OVERLAP,
        "separator": UTTERANCE_SEPARATOR,
        "num_utterances": len(utterances),
        "num_chunks": len(chunks),
    },
    "utterances": [{"utterance_index": i, "text": u} for i, u in enumerate(utterances)],
    "chunks": chunks,
}

chunk_result

# 저장용
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# output_path = OUTPUT_DIR / f"{meeting_id}_chunks.json"
# output_path.write_text(json.dumps(chunk_result, ensure_ascii=False, indent=2), encoding="utf-8")
# print("저장 완료:", output_path)

In [ ]:
# --------------------------------------------------------------------------------------------------
# 참고) Chunk_size / Chunk_overlap 조합별 결과 비교 (글자 수 기준)
# --------------------------------------------------------------------------------------------------
rows = []
for size, overlap in COMPARE_CONFIGS:
    cs = chunk_utterances(utterances, size, overlap)
    chars = [c["metadata"]["char_count"] for c in cs]
    utts = [c["metadata"]["utterance_end"] - c["metadata"]["utterance_start"] + 1 for c in cs]
    rows.append({"chunk_size": size, "chunk_overlap": overlap, "num_chunks": len(cs),
                 "avg_chars": round(sum(chars) / len(chars)), "min_chars": min(chars),
                 "avg_utterances": round(sum(utts) / len(utts), 1)})
display(pd.DataFrame(rows))

## 00-02. 주제 경계 후보 탐지

#### 경계 후보 탐지 설계: LLM 경계 기준에 맞춘 청크 간 유사도 비교

- 후속 단계인 LLM 경계 검토에서 평가하는 경계 기준은 **"회의 참여자가 무엇에 대해 결론을 내리려 하는가(논의 대상)"가 바뀌는 지점**
- 같은 대상 안에서 측면만 바뀌거나, 잠깐 다른 이야기를 하다 돌아오는 경우는 경계가 아님으로 판단하기 때문에, 후보 탐지 단계에서도 **단어나 화제의 작은 흔들림이 아닌 논의 대상 단위의 변화**에 반응해야 함

| **설계** | **이유 (LLM 기준과의 연결)** |
|---|---|
| **① pooled 임베딩**: 발화별 BGE-M3 임베딩 → 짧은 맞장구(`네.`, `음.` 등 `FILLER_MAX_CHARS` 이하) 제외 → **글자 수 가중 평균**으로 청크 벡터 생성 | 이 회의 발화의 1/3이 맞장구이기 때문에, 청크 전체를 그대로 임베딩(`direct`)하면 이런 발화 때문에 청크끼리 비슷해지거나 잡음이 생김. 논의 대상은 내용 발화에 담겨 있으므로 내용 발화의 비중을 키움. |
| **② 블록 비교 (`BLOCK_SIZE=2`)**: 경계 g의 왼쪽 청크 2개 평균 vs 오른쪽 청크 2개 평균 | 바로 옆 청크 1개끼리 비교하면 '측면 변화'에도 유사도가 크게 흔들릴 가능성 존재. 약 400자 단위로 비교해야 '논의 대상' 수준의 변화가 남음. 측면 변화나 잠깐의 잡담은 블록 안에서 평균되어 약해지도록 함. |
| **③ depth score**: 유사도 곡선에서 양쪽 봉우리 대비 얼마나 깊게 꺼졌는지 (TextTiling) | 같은 제품을 다루는 회의라 유사도 절대값은 전 구간에서 높음(0.87~0.94). 절대 임계값 대신 **주변보다 상대적으로 꺼진 지점**을 찾도록 함. |
| **④ 재현율 우선 임계값**: `depth ≥ mean + DEPTH_STD_K·std` (기본 K=0) | 최종 판정은 LLM이 하며, 후보 단계에서 놓친 경계는 되살릴 수 없으므로 후보를 다소 넉넉히 뽑고, 측면 변화 같은 오탐은 LLM 주제 경계 검증 단계에서 하도록 함. |
| **⑤ 경계 위치**: 두 청크가 겹치는 발화 구간의 가운데 발화 | 오버랩 구간은 양쪽 청크에 모두 속하므로 그 가운데를 경계 발화 위치(`boundary_position`)로 설정. |

In [ ]:
# 01. 수행 결과값 확인
chunk_result

In [ ]:
# --------------------------------------------------------------------------------------------------
# 2-0. 경계 후보 탐지 설정
# 경로/파라미터는 해당 셀에서만 수정
# BGE-M3 최초 실행 시 모델 다운로드(약 2.3GB) 시간이 걸림
# --------------------------------------------------------------------------------------------------
# EMBEDDING_MODEL = "alphabrothers/alpha-sts-v0"
DEVICE = None                    # None: 자동 선택 (GPU 있으면 cuda), 또는 "cpu" / "cuda"
EMBED_BATCH_SIZE = 16

# 01-1 실험에서 저장한 문장 단위 임베딩 캐시 (texts.json + vectors.npy) → 같은 문장은 다시 임베딩하지 않음
EMBED_CACHE_DIR = BASE_DIR / "outputs" / "experiment" / "embeddings"
EMBED_CACHE_MODELS = {
    # 모델 ID: (캐시 폴더 이름, 로딩 옵션, encode 옵션) - 01-1과 같은 설정이어야 캐시 값과 일치
    "BAAI/bge-m3":                             ("bge-m3", {}, {}),
    "alphabrothers/alpha-sts-v0":              ("alpha-sts", {}, {}),
    "jinaai/jina-embeddings-v5-text-small":    ("jina-v5", {"trust_remote_code": True}, {"task": "text-matching"}),
    "Gameselo/STS-multilingual-mpnet-base-v2": ("mpnet-sts", {}, {}),
}

EMBEDDING_MODE = "pooled"       ## "pooled": 발화 임베딩 가중 평균 / "direct": 청크 전체 텍스트 임베딩 (청크 하나를 단순 청크 전체를 임베딩할 것인가, 청크 내 발화 문장 각각을 임베딩한 후 가중 합으로 계산할 것인가의 차이)
                                 # pooled 설명 추가)
                                 #  > 청크 전체를 한 번에 임베딩하지 않고, 발화마다 따로 임베딩한 뒤 그 벡터들의 평균을 내서 청크 벡터를 만드는 방법
                                 #  > 수치적 수식은 "가중 산술 평균"을 활용하며, 가중치와 맞장구 제외한 평균 측정 방식은 직접 정한 것
FILLER_MAX_CHARS = 4            ## pooled에서 이 글자 수 이하 발화는 제외 -> 맞장구 제외 (01-1 실험과 같은 값)
                                 # 4로 설정할 경우, "네.", "음.", "좋아요."는 제외 / "그러니까요."는 포함됨
BLOCK_SIZE = 2                  ## 청크 경계마다 유사도 계산 시 경계 양쪽에서 비교할 청크 수 (경계 g의 왼쪽 청크 N개 평균과 오른쪽 청크 N개 평균을 비교)
                                 # > 즉, 경계 양쪽을 얼마나 넓게 볼 것인지 지정
                                 # * 참고) chunk가 n개면 gap은 n-1개
                                 # ex) gap 19(chunk 19|chunk 20)는 [chunk 18, 19]의 평균과 [chunk 20, 21]의 평균의 코사인 유사도 계산
DEPTH_STD_K = 0.0               ## 얼마나 깊게 꺼져야 후보로 인정할 것인지 지정
                                 # 후보 임계값 = depth 평균 + K * 표준편차 (낮출수록 후보 증가)
                                 # -> depth가 임계값 이상인 지점만 후보가 됨
MIN_GAP_DISTANCE = 1            ## 너무 붙어 있는 후보는 하나만
                                 # 후보 간 최소 간격 (청크 경계 단위, 1이면 제한 없음)
                                 # 즉, 해당 변수 값이 2이며, 후보가 gap =5, 6으로 붙어있을 경우, overlap의 영향으로 같은 경계가 후보 2개로 잡힐 가능성을 고려하여, 5,6 중 depth 값이 작은 경우, 작은 gap을 버림
                                 #     (LLM의 검증 횟수 줄이기 위함)


CONTEXT_CHARS = 300              # LLM에 넘길 경계 앞/뒤 문맥 글자 수 (각각, 약 청크 1개)

BOUNDARY_OUTPUT_DIR = BASE_DIR / "outputs" / "boundaries"

## (참고) 정답(GoldenSet) 기반 평가. 탐지 입력으로는 사용하지 않음. 평가하지 않으려면 None
## GoldenSet의 boundary_positions는 1부터 센 번호 → 노트북에서는 1을 빼서 0부터 센 발화 번호(utterance_index)로 비교
GOLD_PATH = BASE_DIR / "dataset" / "GoldenSet" / "hj" / f"{meeting_id}.json"
EVAL_TOLERANCE = 8               # 평가 허용 오차 : 정답 경계와 ±N발화 이내면 맞춘 것으로 간주
                                 #                즉, 후보 위치가 정답 경계에서 ±8발화 이내면 맞춘 것으로 카운트

In [ ]:
# --------------------------------------------------------------------------------------------------
# 2-1. 발화 임베딩 불러오기 (01-1 실험의 문장 단위 캐시 재사용)
# - EMBED_CACHE_DIR/<모델>/texts.json + vectors.npy에서 문장별 벡터를 찾아 사용 → 모델을 새로 돌리지 않음
# - 캐시에 없는 문장이 있을 때만 모델을 불러와 그 문장만 임베딩 (캐시 파일은 수정하지 않음)
# - direct 모드(청크 전체 임베딩)는 캐시가 없으므로 EMBEDDING_MODE = "direct"일 때만 모델로 계산
# 모든 벡터는 L2 정규화 → 내적 = 코사인 유사도
# --------------------------------------------------------------------------------------------------
import numpy as np
from sentence_transformers import SentenceTransformer

cache_name, load_kwargs, encode_kwargs = EMBED_CACHE_MODELS[EMBEDDING_MODEL]
_embed_model = None


def get_model():
    # 모델이 실제로 필요할 때만 한 번 불러옴
    global _embed_model
    if _embed_model is None:
        _embed_model = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE, **load_kwargs)
    return _embed_model


def embed(texts):
    return get_model().encode(texts, batch_size=EMBED_BATCH_SIZE, normalize_embeddings=True,
                              convert_to_numpy=True, show_progress_bar=True, **encode_kwargs)

# 정규화
def l2norm(v):
    return v / np.linalg.norm(v, axis=-1, keepdims=True)


def load_cached_embeddings(texts):
    # 캐시에 있는 문장은 저장된 벡터를, 없는 문장만 새로 임베딩해서 texts 순서대로 반환
    cache_dir = EMBED_CACHE_DIR / cache_name
    if (cache_dir / "texts.json").is_file() and (cache_dir / "vectors.npy").is_file():
        cached_texts = json.loads((cache_dir / "texts.json").read_text(encoding="utf-8"))
        cached_vectors = np.load(cache_dir / "vectors.npy")
    else:
        print(f"캐시 없음: {cache_dir} → 전체를 새로 임베딩")
        cached_texts, cached_vectors = [], None
    lookup = {t: i for i, t in enumerate(cached_texts)}
    missing = list(dict.fromkeys(t for t in texts if t not in lookup))
    new = dict(zip(missing, embed(missing))) if missing else {}
    vectors = np.array([cached_vectors[lookup[t]] if t in lookup else new[t] for t in texts], dtype=np.float32)
    print(f"[{cache_name}] 발화 {len(texts)}개 중 캐시 사용 {sum(t in lookup for t in texts)}개 / "
          f"새로 임베딩 {len(missing)}개 (서로 다른 문장 기준)")
    return l2norm(vectors)


utt_texts = [u["text"] for u in chunk_result["utterances"]] # 발화문 텍스트 (문장 단위 저장값 각각)
utt_chars = np.array([count_chars(t) for t in utt_texts])   # 발화별 글자 수
utt_embeddings = load_cached_embeddings(utt_texts)          # 발화문 내 문장 단위 임베딩 (캐시)  # (발화 수, 차원)

# 발화문 청크 블럭 임베딩 (direct 모드에서만 계산)                  # (청크 수, 차원)
direct_embeddings = embed([c["text"] for c in chunk_result["chunks"]]) if EMBEDDING_MODE == "direct" else None


# 청크마다 청크에 속 한 발화들의 임베딩을 글자 수로 가중 평균하여, 청크 하나당 벡터 하나를 만드는 함수
def pooled_chunk_embeddings(chunks, filler_max_chars):
    vectors = []
    for c in chunks:
        m = c["metadata"]
        idx = np.arange(m["utterance_start"], m["utterance_end"] + 1)
        content = idx[utt_chars[idx] > filler_max_chars]           # 맞장구 걸러낸 발화 번호 (현재 발화 문장이 맞장구(ex.그러니까)에 포함되지 않을 경우만 저장)
        idx = content if len(content) else idx                       # 전부 맞장구면 그대로 사용 )평균 낼 벡터가 없기 때문에)
        weights = utt_chars[idx][:, None]                            # 가중치(글자 수)를 꺼내고 곱셈을 할 수 있는 모양으로 변환 (발화 문장 길이가 길 수록 높은 비중 차지)
        vectors.append((utt_embeddings[idx] * weights).sum(axis=0))  # 가중 합 계산해 리스트에 넣음
    return l2norm(np.array(vectors))                                 # 청크별 가중합(벡터) 쌓아 행렬로 만든 후 길이 1로 정규화 (이후 아래 단계에서 block_similarities에서 두 벡터를 내적한 값으로 코사인 유사도를 만들기 위함)


embeddings = pooled_chunk_embeddings(chunk_result["chunks"], FILLER_MAX_CHARS)
chunk_embeddings = embeddings if EMBEDDING_MODE == "pooled" else direct_embeddings
print("발화 임베딩:", utt_embeddings.shape, "/ 청크 임베딩:", chunk_embeddings.shape, f"({EMBEDDING_MODE})")

In [ ]:
# --------------------------------------------------------------------------------------------------
# 2-2. 청크 경계별 블록 유사도 → depth score → 경계 후보 선택
# gap g = chunk g 와 chunk g+1 사이 (총 청크 수(n) - 1개)
# --------------------------------------------------------------------------------------------------
# 청크별(gap) 유사도 측정
# -> gap 마다 왼쪽 gap과 오른쪽 gap의 코사인 유사도를 계산해 gap 개수만큼의 유사도 배열 돌려줌
def block_similarities(emb, block_size):
    sims = []

    for g in range(len(emb) - 1):
        # * mean(axis=0)을 통해 청크 벡터들을 세로 방향으로 평균 내도록 함
        left = l2norm(emb[max(0, g - block_size + 1):g + 1].mean(axis=0)) ## 왼쪽 블록 벡터 계산 및 정규화
                                                                           # emb[시작:g+1]을 의미하며, block_size로 인해 단순히 g-1, g+1을 할 수 없음
                                                                           # +) max(0,~) : 회의 맨 앞에서 시작하는 번호(g=0)가 음수가 되지 않도록 함 (시작을 0으로 고정)
                                                                           
        right = l2norm(emb[g + 1:g + 1 + block_size].mean(axis=0))        ## 오른쪽 블록 벡터 계산 및 정규화
    
        sims.append(float(left @ right))                                   # 두 청크의 코사인 유사도 계산 (@ : 내적)
                                                                           # 1에 가까울수록 유사하다는 의미    
    return np.array(sims)


# 각 지점에서 왼쪽/오른쪽으로 유사도가 오르는 동안 따라가 찾은 봉우리 높이와의 차이 합 계산
# -> 해당 값(depth)으로 추후 경계 후보 선정에 활용 (depth >= 임계값 and depth > 0인 gap을 경계 후보로 정할 때)
def depth_scores(sims):
    depths = []
    for i in range(len(sims)):
        j, left_peak = i, sims[i]
        while j > 0 and sims[j - 1] >= left_peak:
            j -= 1
            left_peak = sims[j]
        j, right_peak = i, sims[i]
        while j < len(sims) - 1 and sims[j + 1] >= right_peak:
            j += 1
            right_peak = sims[j]
        depths.append((left_peak - sims[i]) + (right_peak - sims[i]))
    return np.array(depths)


# depth를 통해 어느 gap을 후보로 뽑을지 선택
def select_candidates(depths, std_k, min_gap_distance):
    threshold = depths.mean() + std_k * depths.std()                 # 임계값 계산
    chosen = []
    for g in np.argsort(-depths):                                    # depth 큰 gap부터 하나씩 (MIN_GAP_DISTANCE 규칙으로, 가까운 경계 후보끼리 충돌 시 depth가 더 큰 쪽을 남기기 위함)
        if depths[g] < threshold or depths[g] <= 0:                  # 기준 미달 시 경계 후보 제외
            break
        if all(abs(g - c) >= min_gap_distance for c in chosen):      # 이미 확정된 후보들과 충분히 떨어져 있을 때만 확정하기 위함 (MIN_GAP_DISTANCE 1인 경우 조건문 모두 통과)
            chosen.append(int(g))
    return sorted(chosen), threshold                                 # gap 번호 순서로 재정렬 후 반환


# 뽑힌 gap을 몇 번 발화 위치로 볼 것인지 정함
# -> 즉, '몇 번 발화부터 새 주제인가?'라는 발화 단위 위치가 필요하기 때문에 해당 함수를 통해 변환
# ex) chunk 19 : 319:337
#     chunk 20 : 335:359
#     -> overlap :335 336 337 -> 336을 발화 경계 구간으로 (오버랩 구간의 가운데 발화 번호 반환)
def boundary_position(chunks, g):
    right_start = chunks[g + 1]["metadata"]["utterance_start"]       # 오른쪽 청크가 시작하는 발화 번호
    left_end = chunks[g]["metadata"]["utterance_end"] + 1            # 왼쪽 청크가 끝난 바로 다음 발화 번호
    return (right_start + left_end) // 2


chunks_ = chunk_result["chunks"]
gap_sims = block_similarities(chunk_embeddings, BLOCK_SIZE)
gap_depths = depth_scores(gap_sims)
candidate_gaps, depth_threshold = select_candidates(gap_depths, DEPTH_STD_K, MIN_GAP_DISTANCE)

gaps_df = pd.DataFrame({
    "gap": range(len(gap_sims)),
    "left_chunk": range(len(gap_sims)),
    "right_chunk": range(1, len(gap_sims) + 1),
    "boundary_position": [boundary_position(chunks_, g) for g in range(len(gap_sims))],
    "similarity": gap_sims.round(4),
    "depth": gap_depths.round(4),
})
gaps_df["is_candidate"] = gaps_df["gap"].isin(candidate_gaps)
print(f"depth 임계값: {depth_threshold:.4f}  →  후보 {len(candidate_gaps)}개 / 전체 청크 경계 {len(gap_sims)}개")
display(gaps_df[gaps_df["is_candidate"]])

In [ ]:
# --------------------------------------------------------------------------------------------------
# +) 유사도 / depth 곡선 시각화 (빨간 점선: 후보, 회색 실선: 정답 경계, GOLD_PATH 있을 때만)
# --------------------------------------------------------------------------------------------------
import matplotlib.pyplot as plt

x = gaps_df["boundary_position"]
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
ax1.plot(x, gap_sims, marker="o", ms=3)
ax1.set_ylabel("block cosine similarity")
ax2.bar(x, gap_depths, width=6)
ax2.axhline(depth_threshold, ls="--", c="gray", lw=1)
ax2.set_ylabel("depth score")
ax2.set_xlabel("utterance index")
for g in candidate_gaps:
    for ax in (ax1, ax2):
        ax.axvline(gaps_df.loc[g, "boundary_position"], c="red", ls=":", lw=1)
if GOLD_PATH is not None and Path(GOLD_PATH).is_file():
    for b in json.loads(Path(GOLD_PATH).read_text(encoding="utf-8"))["boundary_positions"]:
        for ax in (ax1, ax2):
            ax.axvline(b - 1, c="gray", alpha=0.5, lw=2)        # 1부터 센 번호 → 0부터 센 번호
ax1.set_title(f"{meeting_id} | {EMBEDDING_MODE}, block={BLOCK_SIZE}, chunk={CHUNK_SIZE}/{CHUNK_OVERLAP}")
plt.tight_layout()
plt.show()

In [ ]:
# --------------------------------------------------------------------------------------------------
# 2-3. LLM 경계 검증 입력 구성
# 경계 후보(boundary_position)를 중앙에 두고 앞/뒤 CONTEXT_CHARS 글자만큼의 발화를 각각 하나의 문맥으로 묶음
# left_context: [.., boundary_position - 1] / right_context: [boundary_position, ..]
# 맞장구도 '종결 신호'(좋아요, 그럼 등)일 수 있으므로 문맥에서는 제외하지 않음
# --------------------------------------------------------------------------------------------------
def collect_context(position, direction, max_chars):
    indices, total = [], 0
    rng = range(position - 1, -1, -1) if direction == "left" else range(position, len(utt_texts))
    for i in rng:
        if indices and total + utt_chars[i] > max_chars:   # 발화 글자 수 합이 max_chars를 넘기 직전까지
            break
        indices.append(i)
        total += utt_chars[i]
    indices = sorted(indices)
    return [{"utterance_index": i, "text": utt_texts[i]} for i in indices]


candidates = []
for rank, g in enumerate(sorted(candidate_gaps, key=lambda g: -gap_depths[g])):
    pos = boundary_position(chunks_, g)
    candidates.append({
        "candidate_id": None,
        "boundary_position": pos,                 # 이 발화부터 새 주제라고 보는 후보 위치
        "gap_index": g,                           # chunk g | chunk g+1
        "left_chunk_index": g,
        "right_chunk_index": g + 1,
        "similarity": round(float(gap_sims[g]), 4),
        "depth_score": round(float(gap_depths[g]), 4),
        "depth_rank": rank + 1,
        "left_context": collect_context(pos, "left", CONTEXT_CHARS),
        "right_context": collect_context(pos, "right", CONTEXT_CHARS),
    })
candidates.sort(key=lambda c: c["boundary_position"])
for i, c in enumerate(candidates):
    c["candidate_id"] = i

boundary_result = {
    "meeting_id": meeting_id,
    "metadata": header,
    "detection": {
        "embedding_model": EMBEDDING_MODEL,
        "embedding_mode": EMBEDDING_MODE,
        "filler_max_chars": FILLER_MAX_CHARS,
        "block_size": BLOCK_SIZE,
        "score": "depth score of block cosine similarity (TextTiling)",
        "depth_std_k": DEPTH_STD_K,
        "depth_threshold": round(float(depth_threshold), 4),
        "min_gap_distance": MIN_GAP_DISTANCE,
        "context_chars": CONTEXT_CHARS,
        "chunking": chunk_result["chunking"],
        "num_gaps": len(gap_sims),
        "num_candidates": len(candidates),
    },
    "candidates": candidates,
}


def format_candidate(c, n_preview=3):
    left = c["left_context"][-n_preview:]
    right = c["right_context"][:n_preview]
    lines = [f"=== [{left[-1]['utterance_index']}] | [{right[0]['utterance_index']}] — 위치 {c['boundary_position']} "
             f"(depth={c['depth_score']}, rank={c['depth_rank']}) ===", "왼쪽 문맥:"]
    lines += [f"[{u['utterance_index']}] {u['text']}" for u in left]
    lines += ["오른쪽 문맥:"] + [f"[{u['utterance_index']}] {u['text']}" for u in right]
    return "\n".join(lines)


print(f"경계 후보 {len(candidates)}개 (미리보기: 경계 앞/뒤 3발화, 실제 JSON에는 앞/뒤 {CONTEXT_CHARS}자)\n")
for c in candidates:
    print(format_candidate(c), end="\n\n")

In [ ]:
# --------------------------------------------------------------------------------------------------
# 2-4. 저장
# 01과 동일하게 기본은 저장하지 않음. 저장하려면 주석 풀고 실행
# "outputs/boundaries" 디렉토리 내 <회의ID>_boundary_candidates.json 파일로 저장됨
# --------------------------------------------------------------------------------------------------
boundary_result

# BOUNDARY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# boundary_path = BOUNDARY_OUTPUT_DIR / f"{meeting_id}_boundary_candidates.json"
# boundary_path.write_text(json.dumps(boundary_result, ensure_ascii=False, indent=2), encoding="utf-8")
# print("저장 완료:", boundary_path)

In [ ]:
# --------------------------------------------------------------------------------------------------
# 참고) 정답(GoldenSet) 기반 후보 탐지 평가 (GOLD_PATH가 있을 때만)
# - 번호는 모두 0부터 센 발화 번호(utterance_index) = GoldenSet 파일의 번호 - 1
# - 정답 경계에서 ±EVAL_TOLERANCE 발화 이내에 후보가 있으면 맞춘 것으로 간주
# - random : 같은 개수의 후보를 청크 경계에서 무작위로 뽑았을 때의 평균 (비교 기준선)
# --------------------------------------------------------------------------------------------------
import random


def load_gold_boundaries(gold_path):
    gold = json.loads(Path(gold_path).read_text(encoding="utf-8"))
    assert gold["utterance_count"] == len(utterances), "정답 파일과 전사문의 발화 수가 다릅니다."
    return sorted(p - 1 for p in gold["boundary_positions"])     # 1부터 센 번호 → 0부터 센 번호


def recall_precision(pred, gold, tol):
    if not pred:
        return 0.0, 0.0
    recall = sum(any(abs(p - b) <= tol for p in pred) for b in gold) / len(gold)
    precision = sum(any(abs(p - b) <= tol for b in gold) for p in pred) / len(pred)
    return recall, precision


def random_baseline(n, gold, tol, trials=2000, seed=0):
    rnd = random.Random(seed)
    pool = gaps_df["boundary_position"].tolist()
    scores = [recall_precision(rnd.sample(pool, n), gold, tol) for _ in range(trials)]
    return tuple(np.mean(scores, axis=0))


# 경계 후보 번호 / 정답 경계 번호 / 맞춘 개수를 한 번에 출력하고, 정답 경계별 표를 돌려줌
def match_report(pred, gold, tol, title):
    pred = sorted(pred)
    rows = []
    for b in gold:
        nearest = min(pred, key=lambda p: abs(p - b)) if pred else None
        dist = abs(nearest - b) if pred else None
        rows.append({"정답 경계": b, "가장 가까운 후보": nearest, "거리(발화)": dist,
                     "맞춤": "O" if pred and dist <= tol else "X"})
    hit = sum(r["맞춤"] == "O" for r in rows)
    near = sum(any(abs(p - b) <= tol for b in gold) for p in pred)
    print(f"[{title}] 허용 오차 ±{tol}발화")
    print(f"  경계 후보 번호 ({len(pred)}개): {pred}")
    print(f"  정답 경계 번호 ({len(gold)}개): {gold}")
    print(f"  맞춘 정답 경계: {hit} / {len(gold)}  (재현율 {hit / len(gold):.2f})")
    print(f"  정답 근처 후보: {near} / {len(pred)}  (정밀도 {near / len(pred) if pred else 0:.2f})")
    print(f"  놓친 정답 경계: {[r['정답 경계'] for r in rows if r['맞춤'] == 'X']}")
    return pd.DataFrame(rows)


if GOLD_PATH is not None and Path(GOLD_PATH).is_file():
    gold_bounds = load_gold_boundaries(GOLD_PATH)
    pred = [c["boundary_position"] for c in candidates]
    report_df = match_report(pred, gold_bounds, EVAL_TOLERANCE, "경계 후보 탐지")
    rr, rp = random_baseline(len(pred), gold_bounds, EVAL_TOLERANCE)
    print(f"  (비교) 같은 개수를 무작위로 뽑았을 때: 재현율 {rr:.2f} / 정밀도 {rp:.2f}")
    display(report_df)
else:
    print("GOLD_PATH 없음 → 평가 생략")

In [ ]:
# --------------------------------------------------------------------------------------------------
# 2-5. 인접 경계 후보 구간 병합 (LLM 검증 수 줄이기)
# - 청크 겹침(CHUNK_OVERLAP) 때문에 같은 주제 전환이 이웃한 후보 여러 개로 잡히는 문제를 줄이기 위함
# - 위치 순으로 후보를 보며, 바로 앞 후보와의 사이 발화 글자 수가 MERGE_GAP_CHARS 이하이고
#   구간 전체 폭이 MERGE_MAX_SPAN_CHARS 이하이면 같은 구간으로 묶음 → 후보를 버리지 않고 구간으로 합침
# - LLM은 이후 "구간 [span_start]~[span_end] 안에서 새 주제가 시작되는 발화가 있는가?"를 판정 (후보 1개짜리 구간은 기존과 동일)
# ex) 후보 54, 62, 67 → 구간 [54]~[67] 하나 → LLM 검증 3회 → 1회
# --------------------------------------------------------------------------------------------------
MERGE_GAP_CHARS = 400           # 이웃한 후보 사이 발화 글자 수가 이 값 이하면 같은 구간으로 묶음 (클수록 구간 수 감소)
MERGE_MAX_SPAN_CHARS = 600      # 한 구간의 최대 폭(첫 후보 ~ 마지막 후보 사이 발화 글자 수), None이면 제한 없음
                                 #  > 제한이 없으면 연쇄로 묶여 900자 이상 구간이 생기고, 한 구간에 정답 경계가 여러 개 들어갈 수 있음


def chars_between(start, end):
    # 발화 start ~ end-1의 글자 수 합
    return int(utt_chars[start:end].sum())


def merge_candidates(candidates, gap_chars, max_span_chars):
    groups = []
    for c in sorted(candidates, key=lambda c: c["boundary_position"]):
        if groups:
            first, last, pos = groups[-1][0]["boundary_position"], groups[-1][-1]["boundary_position"], c["boundary_position"]
            if chars_between(last, pos) <= gap_chars and (max_span_chars is None or chars_between(first, pos) <= max_span_chars):
                groups[-1].append(c)                                  # 앞 구간에 이어 붙임
                continue
        groups.append([c])                                            # 새 구간 시작
    return groups


def build_span(span_id, group):
    start, end = group[0]["boundary_position"], group[-1]["boundary_position"]
    return {
        "span_id": span_id,
        "span_start": start,                                          # [span_start]~[span_end] 사이 어딘가에서 새 주제가 시작된다고 보는 구간
        "span_end": end,
        "span_chars": chars_between(start, end),
        "member_candidate_ids": [c["candidate_id"] for c in group],
        "member_positions": [c["boundary_position"] for c in group],
        "max_depth_score": max(c["depth_score"] for c in group),
        "left_context": collect_context(start, "left", CONTEXT_CHARS),                            # 구간 시작 이전 문맥
        "span_utterances": [{"utterance_index": i, "text": utt_texts[i]} for i in range(start, end)],  # 구간 안 발화
        "right_context": collect_context(end, "right", CONTEXT_CHARS),                            # 구간 끝 이후 문맥
    }


merged_groups = merge_candidates(candidates, MERGE_GAP_CHARS, MERGE_MAX_SPAN_CHARS)
merged_spans = [build_span(i, g) for i, g in enumerate(merged_groups)]

merged_result = {
    "meeting_id": meeting_id,
    "metadata": header,
    "detection": {**boundary_result["detection"], "merge_gap_chars": MERGE_GAP_CHARS,
                  "merge_max_span_chars": MERGE_MAX_SPAN_CHARS, "num_spans": len(merged_spans)},
    "spans": merged_spans,
}

print(f"경계 후보 {len(candidates)}개 → 병합 구간 {len(merged_spans)}개 "
      f"(MERGE_GAP_CHARS={MERGE_GAP_CHARS}, MERGE_MAX_SPAN_CHARS={MERGE_MAX_SPAN_CHARS})")
display(pd.DataFrame([{k: s[k] for k in ("span_id", "span_start", "span_end", "span_chars", "member_positions", "max_depth_score")}
                      for s in merged_spans]))

In [ ]:
# --------------------------------------------------------------------------------------------------
# 참고) 구간 병합 결과 평가 (GOLD_PATH가 있을 때만, 02 평가 셀을 먼저 실행)
# - 정답 경계가 [span_start - EVAL_TOLERANCE, span_end + EVAL_TOLERANCE] 안에 있으면 그 구간이 맞춘 것으로 간주
#   (후보 1개짜리 구간은 기존 ±EVAL_TOLERANCE 기준과 같음)
# - 마지막 표: 병합 설정 조합별 구간 수와 맞춘 정답 수 → 목표(구간 TARGET_SPANS개, 놓친 정답 MAX_MISSED_GOLD개 이하) 충족 여부
# --------------------------------------------------------------------------------------------------
TARGET_SPANS = (10, 15)         # LLM이 검증할 구간 수 목표 범위
MAX_MISSED_GOLD = 2             # 허용할 놓친 정답 경계 수
SWEEP_GAP_CHARS = [200, 250, 300, 350, 400, 500]
SWEEP_MAX_SPAN_CHARS = [None, 600, 800, 1000]


def golds_in_span(start, end, gold, tol):
    return [b for b in gold if start - tol <= b <= end + tol]


def span_label(start, end):
    return f"{start}" if start == end else f"{start}~{end}"


if GOLD_PATH is not None and Path(GOLD_PATH).is_file():
    tol = EVAL_TOLERANCE
    ranges = [(s["span_start"], s["span_end"]) for s in merged_spans]
    hit = [b for b in gold_bounds if any(golds_in_span(s, e, [b], tol) for s, e in ranges)]
    near = sum(bool(golds_in_span(s, e, gold_bounds, tol)) for s, e in ranges)
    before_hit = sum(any(abs(c["boundary_position"] - b) <= tol for c in candidates) for b in gold_bounds)

    print(f"[구간 병합] 허용 오차 ±{tol}발화")
    print(f"  경계 후보 구간 ({len(ranges)}개): {[span_label(s, e) for s, e in ranges]}")
    print(f"  정답 경계 번호 ({len(gold_bounds)}개): {gold_bounds}")
    print(f"  맞춘 정답 경계: {len(hit)} / {len(gold_bounds)}  (재현율 {len(hit) / len(gold_bounds):.2f})")
    print(f"  정답 근처 구간: {near} / {len(ranges)}  (정밀도 {near / len(ranges):.2f})")
    print(f"  놓친 정답 경계: {[b for b in gold_bounds if b not in hit]}")
    print(f"  (비교) 병합 전: 후보 {len(candidates)}개 / 맞춘 정답 경계 {before_hit} / {len(gold_bounds)}")

    # 구간별: 구간 안(±tol)에 들어온 정답 경계 → 정답이 2개 이상이면 LLM이 구간 안에서 경계를 여러 개 찾아야 함
    display(pd.DataFrame([{"span_id": s["span_id"], "구간": span_label(s["span_start"], s["span_end"]),
                           "span_chars": s["span_chars"], "묶인 후보": s["member_positions"],
                           "구간 내 정답": golds_in_span(s["span_start"], s["span_end"], gold_bounds, tol)}
                          for s in merged_spans]))

    # 병합 설정 조합별 비교 (현재 설정은 is_current=True)
    rows = []
    for gap_chars in SWEEP_GAP_CHARS:
        for max_span in SWEEP_MAX_SPAN_CHARS:
            groups = merge_candidates(candidates, gap_chars, max_span)
            rr = [(g[0]["boundary_position"], g[-1]["boundary_position"]) for g in groups]
            n_hit = sum(any(golds_in_span(s, e, [b], tol) for s, e in rr) for b in gold_bounds)
            rows.append({"MERGE_GAP_CHARS": gap_chars, "MERGE_MAX_SPAN_CHARS": max_span or "없음", "num_spans": len(rr),
                         "맞춘 정답": f"{n_hit} / {len(gold_bounds)}", "놓친 정답 수": len(gold_bounds) - n_hit,
                         "최대 구간 폭(자)": max(chars_between(s, e) for s, e in rr),
                         "정답 2개+ 구간 수": sum(len(golds_in_span(s, e, gold_bounds, tol)) >= 2 for s, e in rr),
                         "목표 충족": TARGET_SPANS[0] <= len(rr) <= TARGET_SPANS[1] and len(gold_bounds) - n_hit <= MAX_MISSED_GOLD,
                         "is_current": (gap_chars, max_span) == (MERGE_GAP_CHARS, MERGE_MAX_SPAN_CHARS)})
    sweep_df = pd.DataFrame(rows).sort_values(["목표 충족", "놓친 정답 수", "num_spans"], ascending=[False, True, True])
    display(sweep_df.reset_index(drop=True))
else:
    print("GOLD_PATH 없음 → 평가 생략")

# 01. 20개 회의록 기준 경계 후보 탐지 - 최적 조합 테스트

- **대상:** GoldenSet/hj의 20개 회의 (전사문: `playground/hyungjin/test/input`)
- **고정값:** 02 섹션과 동일 → `CHUNK_SIZE=200`, `CHUNK_OVERLAP=100`, `FILLER_MAX_CHARS=4`, `BLOCK_SIZE=2`, `MIN_GAP_DISTANCE=1`, pooled 임베딩, 허용 오차 ±8발화
- **조정값:** 임베딩 모델 2 × `DEPTH_STD_K` 2 × `MERGE_GAP_CHARS` 3 × `MERGE_MAX_SPAN_CHARS` 3 = **36개 조합**
- **임베딩:** 01-1 실험의 문장 단위 캐시를 그대로 사용 (새로 임베딩하지 않음)
- **평가 기준 (회의마다):**
  1. 병합 구간 수가 10~15개
  2. 놓친 정답 경계가 2개 이하
  3. 정답 경계가 2개 이상 들어간 구간(**복수 정답 구간**)이 적을수록 좋음
- **최적 조합:** 1·2를 모두 만족한 회의 수 ↑ → 복수 정답 구간 수 합 ↓ → 놓친 정답 수 합 ↓ → 평균 구간 수 ↓ 순으로 정렬한 1위
- 번호는 모두 0부터 센 발화 번호 (= GoldenSet 파일의 번호 - 1)

In [ ]:
# --------------------------------------------------------------------------------------------------
# 1-0. 최적 조합 테스트 설정
# 01·02 섹션의 함수(load_transcript, chunk_utterances, block_similarities, depth_scores, select_candidates,
# boundary_position, l2norm)를 사용하므로 02 섹션의 2-2 셀까지 먼저 실행할 것
# 02 섹션의 단일 회의 설정값과 섞이지 않도록 이 테스트의 변수는 OPT_로 시작
# --------------------------------------------------------------------------------------------------
import itertools
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer


# ── 경로 ──────────────────────────────────────────────
BASE_DIR = Path.cwd().parent
OPT_TRANSCRIPT_DIR = BASE_DIR.parent / "hyungjin" / "test" / "input"       # 20개 회의 전사문
OPT_GOLD_DIR = BASE_DIR / "dataset" / "GoldenSet" / "hj"                    # 20개 회의 정답 경계
OPT_EMBED_CACHE_DIR = BASE_DIR / "outputs" / "experiment" / "embeddings"    # 01-1 실험의 문장 단위 임베딩 캐시
OPT_OUTPUT_DIR = BASE_DIR / "outputs" / "experiment"
OPT_MEETING_IDS = None          # None: GoldenSet/hj의 회의 전체(20개) / 일부만 보려면 ["TS3003b", "TS3003c"]

# ── 고정값 ────────────────────────────────────────────
OPT_CHUNK_SIZE = 200
OPT_CHUNK_OVERLAP = 100
OPT_FILLER_MAX_CHARS = 4
OPT_BLOCK_SIZE = 2
OPT_MIN_GAP_DISTANCE = 1
OPT_TOLERANCE = 8               # 정답 경계가 [구간 시작 - 8, 구간 끝 + 8] 안에 있으면 맞춘 것으로 간주

# ── 조정값 (조합 = 모델 × DEPTH_STD_K × MERGE_GAP_CHARS × MERGE_MAX_SPAN_CHARS) ──
OPT_EMBEDDING_MODELS = {        # 모델 ID: 캐시 폴더 이름
    "BAAI/bge-m3": "bge-m3",
    "alphabrothers/alpha-sts-v0": "alpha-sts",
}
OPT_DEPTH_STD_K = [0.0, 0.2]
OPT_MERGE_GAP_CHARS = [200, 300, 400]
OPT_MERGE_MAX_SPAN_CHARS = [600, 700, 800]

# ── 전사문 형식 ───────────────────────────────────────
HEADER_LINES = 3            # 상단 메타데이터 줄 수 (회의 대주제 / 참여자 / 일시) → 청킹 제외
UTTERANCE_SEPARATOR = "\n"  # 청크 안에서 발화를 이어 붙일 구분자


# ── 목표 ──────────────────────────────────────────────
OPT_TARGET_SPANS = (10, 15)     # 회의당 병합 구간 수 목표 범위
OPT_MAX_MISSED = 2              # 회의당 허용할 놓친 정답 경계 수

# ── 상세 표 ───────────────────────────────────────────
OPT_SELECTED_COMBO = None       # None: 최적 조합 / 직접 지정: ("alpha-sts", 0.0, 400, 600) = (모델 캐시 이름, DEPTH_STD_K, GAP, MAX_SPAN)
OPT_DETAIL_MEETINGS = None      # 구간 상세 표를 볼 회의 (None: 전체 / 일부: ["TS3003c"])

In [ ]:
# --------------------------------------------------------------------------------------------------
# 1-1. 20개 회의 불러오기 → 청킹 → 캐시 임베딩으로 청크 경계(gap)별 depth 계산
# 모델·회의마다 depth까지는 한 번만 계산 (DEPTH_STD_K, 병합 설정과 무관)
# --------------------------------------------------------------------------------------------------
def load_transcript(path, header_lines):
    lines = path.read_text(encoding="utf-8-sig").splitlines()
    header = {}
    for line in lines[:header_lines]:
        key, _, value = line.partition(":")
        header[key.strip()] = value.strip()
    utterances = [line.strip() for line in lines[header_lines:] if line.strip()]
    return header, utterances

def count_chars(text):
    # 글자 수 = len(text) → 공백과 발화 사이 줄바꿈(UTTERANCE_SEPARATOR)도 1자로 셈
    return len(text)

def chunk_utterances(utterances, chunk_size, chunk_overlap, sep=UTTERANCE_SEPARATOR):
    chunks = []
    start = 0
    while start < len(utterances):
        # 1) chunk_size를 넘기 직전까지 발화를 쌓는다 (발화 1개가 chunk_size보다 길면 그 발화 하나로 청크 구성)
        end = start + 1
        while end < len(utterances) and count_chars(sep.join(utterances[start:end + 1])) <= chunk_size:
            end += 1
        text = sep.join(utterances[start:end])
        chunks.append({
            "text": text,
            "metadata": {
                "chunk_index": len(chunks),
                "char_count": count_chars(text),
                "utterance_start": start,    # 포함
                "utterance_end": end - 1,    # 포함
            },
        })
        if end == len(utterances):
            break
        # 2) 청크 끝에서 chunk_overlap 이내의 발화를 다음 청크 시작으로 (항상 앞으로 진행)
        next_start = end
        while next_start - 1 > start and count_chars(sep.join(utterances[next_start - 1:end])) <= chunk_overlap:
            next_start -= 1
        start = next_start
    return chunks

def l2norm(v):
    return v / np.linalg.norm(v, axis=-1, keepdims=True)

def pooled_embeddings(chunks, utt_emb, utt_chars, filler_max_chars):
    # 2-1 셀의 pooled_chunk_embeddings와 같은 계산 (회의별 발화 임베딩·글자 수를 인자로 받음)
    vectors = []
    for c in chunks:
        m = c["metadata"]
        idx = np.arange(m["utterance_start"], m["utterance_end"] + 1)
        content = idx[utt_chars[idx] > filler_max_chars]
        idx = content if len(content) else idx
        vectors.append((utt_emb[idx] * utt_chars[idx][:, None]).sum(axis=0))
    return l2norm(np.array(vectors))

def block_similarities(emb, block_size):
    sims = []

    for g in range(len(emb) - 1):
        # * mean(axis=0)을 통해 청크 벡터들을 세로 방향으로 평균 내도록 함
        left = l2norm(emb[max(0, g - block_size + 1):g + 1].mean(axis=0)) ## 왼쪽 블록 벡터 계산 및 정규화
                                                                           # emb[시작:g+1]을 의미하며, block_size로 인해 단순히 g-1, g+1을 할 수 없음
                                                                           # +) max(0,~) : 회의 맨 앞에서 시작하는 번호(g=0)가 음수가 되지 않도록 함 (시작을 0으로 고정)
                                                                           
        right = l2norm(emb[g + 1:g + 1 + block_size].mean(axis=0))        ## 오른쪽 블록 벡터 계산 및 정규화
    
        sims.append(float(left @ right))                                   # 두 청크의 코사인 유사도 계산 (@ : 내적)
                                                                           # 1에 가까울수록 유사하다는 의미    
    return np.array(sims)


def depth_scores(sims):
    depths = []
    for i in range(len(sims)):
        j, left_peak = i, sims[i]
        while j > 0 and sims[j - 1] >= left_peak:
            j -= 1
            left_peak = sims[j]
        j, right_peak = i, sims[i]
        while j < len(sims) - 1 and sims[j + 1] >= right_peak:
            j += 1
            right_peak = sims[j]
        depths.append((left_peak - sims[i]) + (right_peak - sims[i]))
    return np.array(depths)


# depth를 통해 어느 gap을 후보로 뽑을지 선택 (2-2 셀과 동일, 1-2 셀에서 사용)
def select_candidates(depths, std_k, min_gap_distance):
    threshold = depths.mean() + std_k * depths.std()                 # 임계값 계산
    chosen = []
    for g in np.argsort(-depths):                                    # depth 큰 gap부터 하나씩
        if depths[g] < threshold or depths[g] <= 0:                  # 기준 미달 시 경계 후보 제외
            break
        if all(abs(g - c) >= min_gap_distance for c in chosen):      # 이미 확정된 후보들과 충분히 떨어져 있을 때만 확정
            chosen.append(int(g))
    return sorted(chosen), threshold


# gap g(chunk g | chunk g+1)를 발화 번호로 변환 (2-2 셀과 동일): 두 청크가 겹치는 발화 구간의 가운데 발화 번호
def boundary_position(chunks, g):
    right_start = chunks[g + 1]["metadata"]["utterance_start"]       # 오른쪽 청크가 시작하는 발화 번호
    left_end = chunks[g]["metadata"]["utterance_end"] + 1            # 왼쪽 청크가 끝난 바로 다음 발화 번호
    return (right_start + left_end) // 2



opt_meeting_ids = OPT_MEETING_IDS or sorted(p.stem for p in OPT_GOLD_DIR.glob("*.json"))
opt_meetings = {}
for mid in opt_meeting_ids:
    _, utts = load_transcript(OPT_TRANSCRIPT_DIR / f"{mid}.txt", HEADER_LINES)
    gold = json.loads((OPT_GOLD_DIR / f"{mid}.json").read_text(encoding="utf-8"))
    assert gold["utterance_count"] == len(utts), f"{mid}: 정답 파일과 전사문의 발화 수가 다릅니다."
    opt_meetings[mid] = {
        "utterances": utts,
        "chars": np.array([len(u) for u in utts]),
        "chunks": chunk_utterances(utts, OPT_CHUNK_SIZE, OPT_CHUNK_OVERLAP),
        "gold": sorted(p - 1 for p in gold["boundary_positions"]),   # 1부터 센 번호 → 0부터 센 번호
    }

opt_gaps = {}   # (캐시 이름, 회의) → {"depths": gap별 depth, "positions": gap별 경계 발화 번호}
for model_id, cache_name in OPT_EMBEDDING_MODELS.items():
    cache_dir = OPT_EMBED_CACHE_DIR / cache_name
    cached_texts = json.loads((cache_dir / "texts.json").read_text(encoding="utf-8"))
    cached_vectors = np.load(cache_dir / "vectors.npy")
    lookup = {t: i for i, t in enumerate(cached_texts)}
    for mid, m in opt_meetings.items():
        missing = [u for u in m["utterances"] if u not in lookup]
        assert not missing, f"[{cache_name}] {mid}: 캐시에 없는 발화 {len(missing)}개 → 01-1 실험의 1-3 셀로 먼저 임베딩"
        utt_emb = l2norm(cached_vectors[[lookup[u] for u in m["utterances"]]].astype(np.float32))
        chunk_emb = pooled_embeddings(m["chunks"], utt_emb, m["chars"], OPT_FILLER_MAX_CHARS)
        sims = block_similarities(chunk_emb, OPT_BLOCK_SIZE)
        opt_gaps[(cache_name, mid)] = {
            "depths": depth_scores(sims),
            "positions": [boundary_position(m["chunks"], g) for g in range(len(sims))],
        }

print(f"회의 {len(opt_meetings)}개 / 모델 {len(OPT_EMBEDDING_MODELS)}개 → depth 계산 완료")
display(pd.DataFrame([{"meeting_id": mid, "발화 수": len(m["utterances"]), "청크 수": len(m["chunks"]),
                       "정답 경계 수": len(m["gold"])} for mid, m in opt_meetings.items()]).set_index("meeting_id").T)

In [ ]:
# --------------------------------------------------------------------------------------------------
# 1-2. 조합 × 회의별 경계 후보 추출 → 구간 병합 → 정답 비교
# 병합 규칙은 2-5 셀과 동일: 이웃 후보 사이 글자 수 ≤ MERGE_GAP_CHARS 이고 구간 폭 ≤ MERGE_MAX_SPAN_CHARS이면 같은 구간
# --------------------------------------------------------------------------------------------------
def merge_positions(positions, utt_chars, gap_chars, max_span_chars):
    # 후보 위치 목록 → 구간 목록 (구간 = 묶인 후보 위치 리스트)
    spans = []
    for p in sorted(positions):
        if spans and int(utt_chars[spans[-1][-1]:p].sum()) <= gap_chars and int(utt_chars[spans[-1][0]:p].sum()) <= max_span_chars:
            spans[-1].append(p)
        else:
            spans.append([p])
    return spans


def golds_in_span(span, gold, tol):
    return [b for b in gold if span[0] - tol <= b <= span[-1] + tol]


def span_label(span):
    return f"{span[0]}" if span[0] == span[-1] else f"{span[0]}~{span[-1]}"


opt_rows = []
for cache_name, std_k, mid in itertools.product(OPT_EMBEDDING_MODELS.values(), OPT_DEPTH_STD_K, opt_meetings):
    m, g = opt_meetings[mid], opt_gaps[(cache_name, mid)]
    chosen, _ = select_candidates(g["depths"], std_k, OPT_MIN_GAP_DISTANCE)
    positions = sorted(g["positions"][x] for x in chosen)
    for gap_chars, max_span in itertools.product(OPT_MERGE_GAP_CHARS, OPT_MERGE_MAX_SPAN_CHARS):
        spans = merge_positions(positions, m["chars"], gap_chars, max_span)
        span_golds = [golds_in_span(s, m["gold"], OPT_TOLERANCE) for s in spans]
        hit = sorted({b for gs in span_golds for b in gs})
        missed = [b for b in m["gold"] if b not in hit]
        opt_rows.append({
            "model": cache_name, "depth_std_k": std_k, "merge_gap_chars": gap_chars, "merge_max_span_chars": max_span,
            "meeting_id": mid,
            "경계 후보 수": len(positions),
            "병합 구간 수": len(spans),
            "정답 경계 번호": m["gold"],
            "경계 후보 번호(병합)": [span_label(s) for s in spans],
            "맞춘 정답": f"{len(hit)} / {len(m['gold'])}",
            "놓친 정답 번호": missed,
            "복수 정답 구간 수": sum(len(gs) >= 2 for gs in span_golds),
            "맞춘 수": len(hit), "정답 수": len(m["gold"]), "놓친 수": len(missed),
            "구간 수 충족": OPT_TARGET_SPANS[0] <= len(spans) <= OPT_TARGET_SPANS[1],
            "놓친 정답 충족": len(missed) <= OPT_MAX_MISSED,
            "spans": spans, "span_golds": span_golds,          # 1-5 구간 상세 표용
        })
opt_df = pd.DataFrame(opt_rows)
opt_df["목표 충족"] = opt_df["구간 수 충족"] & opt_df["놓친 정답 충족"]
print(f"조합 {len(opt_df) // len(opt_meetings)}개 × 회의 {len(opt_meetings)}개 = {len(opt_df)}개 결과")

In [ ]:
# --------------------------------------------------------------------------------------------------
# 1-3. 조합별 결과 요약 (20개 회의 합계/평균) → 최적 조합 선택
# - 목표 충족 회의 : 구간 수 10~15개 + 놓친 정답 2개 이하를 모두 만족한 회의 수
# - 복수 정답 구간 : 정답 경계가 2개 이상 들어간 구간 수의 합 (적을수록 좋음)
# --------------------------------------------------------------------------------------------------
COMBO_COLS = ["model", "depth_std_k", "merge_gap_chars", "merge_max_span_chars"]

opt_summary = opt_df.groupby(COMBO_COLS).agg(**{
    "목표 충족 회의": ("목표 충족", "sum"),
    "구간 수 충족 회의": ("구간 수 충족", "sum"),
    "놓친 정답 충족 회의": ("놓친 정답 충족", "sum"),
    "평균 경계 후보 수": ("경계 후보 수", "mean"),
    "평균 병합 구간 수": ("병합 구간 수", "mean"),
    "최소 구간 수": ("병합 구간 수", "min"),
    "최대 구간 수": ("병합 구간 수", "max"),
    "맞춘 수": ("맞춘 수", "sum"),
    "정답 수": ("정답 수", "sum"),
    "놓친 정답 합": ("놓친 수", "sum"),
    "회의별 최대 놓친 수": ("놓친 수", "max"),
    "복수 정답 구간 합": ("복수 정답 구간 수", "sum"),
}).reset_index()
opt_summary.insert(len(COMBO_COLS) + 7, "맞춘 정답", opt_summary["맞춘 수"].astype(str) + " / " + opt_summary["정답 수"].astype(str))
opt_summary["재현율"] = (opt_summary["맞춘 수"] / opt_summary["정답 수"]).round(3)
opt_summary = opt_summary.drop(columns=["맞춘 수", "정답 수"]).round({"평균 경계 후보 수": 1, "평균 병합 구간 수": 1})
opt_summary = opt_summary.sort_values(["목표 충족 회의", "복수 정답 구간 합", "놓친 정답 합", "평균 병합 구간 수"],
                                      ascending=[False, True, True, True]).reset_index(drop=True)
opt_summary.index += 1   # 순위

best_combo = tuple(v.item() if hasattr(v, "item") else v for v in opt_summary.loc[1, COMBO_COLS])
selected_combo = tuple(OPT_SELECTED_COMBO) if OPT_SELECTED_COMBO else best_combo
print(f"회의 {len(opt_meetings)}개 기준 / 목표: 구간 {OPT_TARGET_SPANS[0]}~{OPT_TARGET_SPANS[1]}개, 놓친 정답 {OPT_MAX_MISSED}개 이하")
print("최적 조합 (model, depth_std_k, merge_gap_chars, merge_max_span_chars):", best_combo)
with pd.option_context("display.max_rows", None):
    display(opt_summary)

# 저장용 (기본은 저장하지 않음, 주석 풀고 실행하면 outputs/experiment에 저장)
# OPT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# opt_summary.to_csv(OPT_OUTPUT_DIR / "merge_combinations_summary.csv", index_label="rank", encoding="utf-8-sig")
# opt_df.drop(columns=["spans", "span_golds"]).to_csv(OPT_OUTPUT_DIR / "merge_combinations_by_meeting.csv", index=False, encoding="utf-8-sig")

In [ ]:
# --------------------------------------------------------------------------------------------------
# 1-4. 선택 조합의 회의별 결과 (OPT_SELECTED_COMBO가 None이면 최적 조합)
# 경계 후보 수 → 병합 구간 수 / 정답 경계 번호 / 병합된 경계 후보 번호 / 맞춘 정답 / 놓친 정답 번호를 한 표로 확인
# --------------------------------------------------------------------------------------------------
sel = opt_df[(opt_df[COMBO_COLS] == pd.Series(selected_combo, index=COMBO_COLS)).all(axis=1)].set_index("meeting_id")
show_cols = ["경계 후보 수", "병합 구간 수", "정답 경계 번호", "경계 후보 번호(병합)", "맞춘 정답", "놓친 정답 번호",
             "복수 정답 구간 수", "목표 충족"]

print("선택 조합 (model, depth_std_k, merge_gap_chars, merge_max_span_chars):", selected_combo)
print(f"목표 충족 회의: {int(sel['목표 충족'].sum())} / {len(sel)}  |  "
      f"맞춘 정답 합계: {sel['맞춘 수'].sum()} / {sel['정답 수'].sum()}  |  "
      f"병합 구간 수: 평균 {sel['병합 구간 수'].mean():.1f}개 (최소 {sel['병합 구간 수'].min()} ~ 최대 {sel['병합 구간 수'].max()})  |  "
      f"복수 정답 구간 합계: {sel['복수 정답 구간 수'].sum()}개")
with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
    display(sel[show_cols])

In [ ]:
# --------------------------------------------------------------------------------------------------
# 1-5. 선택 조합의 회의별 구간 상세 (OPT_DETAIL_MEETINGS가 None이면 전체 회의)
# 구간 번호 / 구간 / 묶인 후보 / 구간 내 정답 → 어떤 후보가 묶였고, 어떤 정답이 포함됐는지 확인
# 놓친 정답은 가장 가까운 구간과의 거리(발화)도 함께 표시
# --------------------------------------------------------------------------------------------------
for mid, r in sel.iterrows():
    if OPT_DETAIL_MEETINGS and mid not in OPT_DETAIL_MEETINGS:
        continue
    missed_info = []
    for b in r["놓친 정답 번호"]:
        nearest = min(r["spans"], key=lambda s: min(abs(b - s[0]), abs(b - s[-1])))
        missed_info.append(f"{b}(가까운 구간 {span_label(nearest)}, {min(abs(b - nearest[0]), abs(b - nearest[-1]))}발화)")
    print(f"=== {mid} | 경계 후보 {r['경계 후보 수']}개 → 병합 구간 {r['병합 구간 수']}개 | 맞춘 정답 {r['맞춘 정답']} | "
          f"놓친 정답 {', '.join(missed_info) or '없음'} ===")
    detail = pd.DataFrame([{"구간 번호": i, "구간": span_label(s), "묶인 후보": s, "묶인 후보 수": len(s),
                            "구간 내 정답": gs, "복수 정답": "O" if len(gs) >= 2 else ""}
                           for i, (s, gs) in enumerate(zip(r["spans"], r["span_golds"]))])
    with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
        display(detail.set_index("구간 번호"))